# Dutch Healthcare Cost Benchmark Analysis Using Vektis Open Data 2023

**Author:** Mahdi Dadgar  
**Dataset:** Vektis Open Databestand Zorgverzekeringswet 2023 — municipality level

## Why I built this project

I developed this project to better understand how healthcare benchmark analysis can be used to identify meaningful regional cost variation.

My background is in scientific research and applied statistics, and I am now transitioning into data science. This project gave me an opportunity to combine those areas by working with real Dutch healthcare data and focusing not only on calculations, but also on fair comparison, interpretation, and responsible communication of results.

## Project objective

The main analytical question is:

> Which Dutch municipalities show higher or lower healthcare costs than expected after accounting for differences in age and sex composition?

This first notebook prepares and validates the data used in the later benchmark analysis.

## Analytical perspective

Raw healthcare costs are not directly comparable across municipalities. A municipality with an older population may naturally have higher healthcare expenditure than one with a younger population.

For that reason, the next notebook uses age–sex standardisation to compare actual municipality-level costs with expected costs derived from national reference rates.

## Important limitations

This project uses public, aggregated municipality-level data. It does not contain:

- patient-level data
- hospital-level or provider-level data
- diagnoses or clinical outcomes
- morbidity or socioeconomic risk factors
- treatment-pathway information

The findings should therefore be interpreted as signals for further investigation, not as proof that a municipality or healthcare provider is efficient or inefficient.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# Detect whether the notebook is being run from the project root
# or from the notebooks folder.
current_path = Path.cwd()
PROJECT_ROOT = current_path.parent if current_path.name == "notebooks" else current_path

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES = PROJECT_ROOT / "outputs" / "tables"

csv_path = (
    DATA_RAW
    / "Vektis Open Databestand Zorgverzekeringswet 2023 - gemeente.csv"
)

if not csv_path.exists():
    raise FileNotFoundError(
        "The Vektis CSV was not found. Place it in data/raw using "
        "the filename documented in data/README.md."
    )

print("Project folders detected successfully.")
print("Input file found:", csv_path.name)

Project folders detected successfully.
Input file found: Vektis Open Databestand Zorgverzekeringswet 2023 - gemeente.csv


## 1. Load and inspect the source data

The source file contains one row per reported municipality–age–sex combination, plus a Vektis rest-category row for records that cannot be assigned to a usable municipality–age–sex cell.

In [2]:
df = pd.read_csv(csv_path, sep=";")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (12989, 31)


,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,kosten_huisarts_overig,kosten_hulpmiddelen,kosten_mondzorg,kosten_paramedische_zorg_fysioth,kosten_paramedische_zorg_overig,kosten_ziekenvervoer_zittend,kosten_ziekenvervoer_liggend,kosten_kraamzorg,kosten_verloskundige_zorg,kosten_grensoverschrijdende_zorg,kosten_eerstelijns_ondersteuning,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz
0,V,5 t/m 9 jaar,AMERSFOORT,4361,4331.92,1993077.92,321453.33,324320.16,112567.49,90834.26,330502.14,133320.18,802215.31,95713.96,624176.93,6676.52,13046.81,0.0,0.00,5701.64,0,0.0,0.0,308714.33,0.0,0.0,0.0,222796.62,0.0,0.0,0.0
1,V,10 t/m 14 jaar,OOST GELRE,749,742.98,212622.21,36365.89,55490.34,17453.58,9359.63,40696.29,34310.67,189190.36,41203.68,25046.79,0.00,3189.04,0.0,0.00,1099.43,0,0.0,0.0,0.00,0.0,0.0,0.0,9088.73,0.0,0.0,0.0
2,V,10 t/m 14 jaar,OOSTERHOUT,1498,1487.53,598245.35,82667.47,113762.71,39852.67,25524.37,60438.20,53450.85,309260.83,88869.47,26446.71,0.00,6962.66,0.0,0.00,2806.82,0,0.0,0.0,6577.83,0.0,0.0,0.0,27837.98,0.0,0.0,0.0
3,V,10 t/m 14 jaar,OOSTSTELLINGWERF,647,643.94,296500.62,38004.44,48648.18,16924.25,13611.31,35866.44,38416.84,98746.00,32716.34,12593.34,322.86,955.86,0.0,433.79,4840.87,0,0.0,0.0,1306.40,0.0,0.0,0.0,6293.08,0.0,0.0,0.0
4,V,10 t/m 14 jaar,OOSTZAAN,245,244.07,63902.89,8518.97,18259.37,6162.90,4042.80,17090.64,842.54,66513.11,12291.22,9157.45,0.00,878.58,0.0,0.00,113.86,0,0.0,0.0,0.00,0.0,0.0,0.0,3284.54,0.0,0.0,0.0


In [3]:
print("Columns:")
for column in df.columns:
    print("-", column)

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values per column:")
display(
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_rows")
)

Columns:
- geslacht
- leeftijdsklasse
- gemeentenaam
- aantal_bsn
- aantal_verzekerdejaren
- kosten_medisch_specialistische_z
- kosten_farmacie
- kosten_huisarts_inschrijftarief
- kosten_huisarts_consult
- kosten_huisarts_mdz
- kosten_huisarts_overig
- kosten_hulpmiddelen
- kosten_mondzorg
- kosten_paramedische_zorg_fysioth
- kosten_paramedische_zorg_overig
- kosten_ziekenvervoer_zittend
- kosten_ziekenvervoer_liggend
- kosten_kraamzorg
- kosten_verloskundige_zorg
- kosten_grensoverschrijdende_zorg
- kosten_eerstelijns_ondersteuning
- kosten_geriatrische_revalidatiez
- kosten_eerstelijnsverblijf
- kosten_verpleging_en_verzorging
- kosten_gzsp
- kosten_integrale_geboortezorg
- kosten_innovatiegelden_ggz
- kosten_overig
- kosten_consulten_ggz
- kosten_intramuraal_verblijf_ggz
- kosten_overige_prestaties_ggz

Data types:


,dtype
geslacht,str
leeftijdsklasse,str
gemeentenaam,str
aantal_bsn,int64
aantal_verzekerdejaren,float64
kosten_medisch_specialistische_z,float64
kosten_farmacie,float64
kosten_huisarts_inschrijftarief,float64
kosten_huisarts_consult,float64
kosten_huisarts_mdz,float64



Missing values per column:


,missing_rows
geslacht,1
leeftijdsklasse,1
gemeentenaam,1
aantal_bsn,0
aantal_verzekerdejaren,0
kosten_medisch_specialistische_z,0
kosten_farmacie,0
kosten_huisarts_inschrijftarief,0
kosten_huisarts_consult,0
kosten_huisarts_mdz,0


## 2. Define dimensions and healthcare cost fields

The three dimensions used for benchmarking are municipality, five-year age class, and sex. All columns beginning with `kosten_` are treated as healthcare cost components.

In [4]:
dimension_cols = [
    "gemeentenaam",
    "leeftijdsklasse",
    "geslacht",
]

cost_cols = [
    column
    for column in df.columns
    if column.startswith("kosten_")
]

print("Dimension columns:", dimension_cols)
print("Number of healthcare cost columns:", len(cost_cols))

display(pd.DataFrame({"cost_column": cost_cols}))

Dimension columns: ['gemeentenaam', 'leeftijdsklasse', 'geslacht']
Number of healthcare cost columns: 26


,cost_column
0,kosten_medisch_specialistische_z
1,kosten_farmacie
2,kosten_huisarts_inschrijftarief
3,kosten_huisarts_consult
4,kosten_huisarts_mdz
5,kosten_huisarts_overig
6,kosten_hulpmiddelen
7,kosten_mondzorg
8,kosten_paramedische_zorg_fysioth
9,kosten_paramedische_zorg_overig


In [5]:
dimension_overview = pd.DataFrame(
    {
        "measure": [
            "Rows",
            "Municipalities",
            "Age classes",
            "Sex categories",
        ],
        "value": [
            len(df),
            df["gemeentenaam"].nunique(dropna=True),
            df["leeftijdsklasse"].nunique(dropna=True),
            df["geslacht"].nunique(dropna=True),
        ],
    }
)

display(dimension_overview)

print(
    "Sex categories:",
    sorted(df["geslacht"].dropna().astype(str).unique()),
)

,measure,value
0,Rows,12989
1,Municipalities,342
2,Age classes,19
3,Sex categories,2


Sex categories: ['M', 'V']


## 3. Clean text fields and engineer analytical variables

Three variables are created:

- `total_healthcare_cost`: sum of all available Vektis cost categories
- `cost_per_insured_year`: total cost divided by insured years
- `specialist_care_cost_per_insured_year`: medical specialist care cost divided by insured years

Using insured years provides a more appropriate denominator than a simple person count when individuals were not insured for the full calendar year.

In [6]:
df_clean = df.copy()

for column in dimension_cols:
    df_clean[column] = (
        df_clean[column]
        .astype("string")
        .str.strip()
    )

df_clean["total_healthcare_cost"] = (
    df_clean[cost_cols]
    .sum(axis=1)
)

df_clean["cost_per_insured_year"] = np.where(
    df_clean["aantal_verzekerdejaren"] > 0,
    (
        df_clean["total_healthcare_cost"]
        / df_clean["aantal_verzekerdejaren"]
    ),
    np.nan,
)

df_clean["specialist_care_cost_per_insured_year"] = np.where(
    df_clean["aantal_verzekerdejaren"] > 0,
    (
        df_clean["kosten_medisch_specialistische_z"]
        / df_clean["aantal_verzekerdejaren"]
    ),
    np.nan,
)

display(df_clean.head())

,geslacht,leeftijdsklasse,gemeentenaam,aantal_bsn,aantal_verzekerdejaren,kosten_medisch_specialistische_z,kosten_farmacie,kosten_huisarts_inschrijftarief,kosten_huisarts_consult,kosten_huisarts_mdz,kosten_huisarts_overig,kosten_hulpmiddelen,kosten_mondzorg,kosten_paramedische_zorg_fysioth,kosten_paramedische_zorg_overig,kosten_ziekenvervoer_zittend,kosten_ziekenvervoer_liggend,kosten_kraamzorg,kosten_verloskundige_zorg,kosten_grensoverschrijdende_zorg,kosten_eerstelijns_ondersteuning,kosten_geriatrische_revalidatiez,kosten_eerstelijnsverblijf,kosten_verpleging_en_verzorging,kosten_gzsp,kosten_integrale_geboortezorg,kosten_innovatiegelden_ggz,kosten_overig,kosten_consulten_ggz,kosten_intramuraal_verblijf_ggz,kosten_overige_prestaties_ggz,total_healthcare_cost,cost_per_insured_year,specialist_care_cost_per_insured_year
0,V,5 t/m 9 jaar,AMERSFOORT,4361,4331.92,1993077.92,321453.33,324320.16,112567.49,90834.26,330502.14,133320.18,802215.31,95713.96,624176.93,6676.52,13046.81,0.0,0.00,5701.64,0,0.0,0.0,308714.33,0.0,0.0,0.0,222796.62,0.0,0.0,0.0,5385117.60,1243.124896,460.091119
1,V,10 t/m 14 jaar,OOST GELRE,749,742.98,212622.21,36365.89,55490.34,17453.58,9359.63,40696.29,34310.67,189190.36,41203.68,25046.79,0.00,3189.04,0.0,0.00,1099.43,0,0.0,0.0,0.00,0.0,0.0,0.0,9088.73,0.0,0.0,0.0,675116.64,908.660583,286.174877
2,V,10 t/m 14 jaar,OOSTERHOUT,1498,1487.53,598245.35,82667.47,113762.71,39852.67,25524.37,60438.20,53450.85,309260.83,88869.47,26446.71,0.00,6962.66,0.0,0.00,2806.82,0,0.0,0.0,6577.83,0.0,0.0,0.0,27837.98,0.0,0.0,0.0,1442703.92,969.865428,402.173637
3,V,10 t/m 14 jaar,OOSTSTELLINGWERF,647,643.94,296500.62,38004.44,48648.18,16924.25,13611.31,35866.44,38416.84,98746.00,32716.34,12593.34,322.86,955.86,0.0,433.79,4840.87,0,0.0,0.0,1306.40,0.0,0.0,0.0,6293.08,0.0,0.0,0.0,646180.62,1003.479548,460.447588
4,V,10 t/m 14 jaar,OOSTZAAN,245,244.07,63902.89,8518.97,18259.37,6162.90,4042.80,17090.64,842.54,66513.11,12291.22,9157.45,0.00,878.58,0.0,0.00,113.86,0,0.0,0.0,0.00,0.0,0.0,0.0,3284.54,0.0,0.0,0.0,211058.87,864.747286,261.821977


## 4. Check structural completeness

With 342 municipalities, 19 age classes, and 2 sex categories, a fully populated dataset would contain 12,996 municipality–age–sex combinations.

The public source does not necessarily contain every possible combination. Missing combinations are documented rather than imputed, because the aggregated data does not support a defensible replacement value.

In [7]:
combination_cols = [
    "gemeentenaam",
    "leeftijdsklasse",
    "geslacht",
]

valid_dimension_rows = (
    df_clean
    .dropna(subset=combination_cols)
    .copy()
)

observed_combinations = (
    valid_dimension_rows[combination_cols]
    .drop_duplicates()
)

full_combination_index = pd.MultiIndex.from_product(
    [
        sorted(valid_dimension_rows["gemeentenaam"].unique()),
        sorted(valid_dimension_rows["leeftijdsklasse"].unique()),
        sorted(valid_dimension_rows["geslacht"].unique()),
    ],
    names=combination_cols,
)

observed_combination_index = pd.MultiIndex.from_frame(
    observed_combinations
)

missing_combinations = (
    full_combination_index
    .difference(observed_combination_index)
    .to_frame(index=False)
)

expected_combination_count = len(full_combination_index)
observed_combination_count = len(observed_combination_index)

print(
    "Expected municipality-age-sex combinations:",
    expected_combination_count,
)
print(
    "Observed municipality-age-sex combinations:",
    observed_combination_count,
)
print(
    "Missing municipality-age-sex combinations:",
    len(missing_combinations),
)

display(missing_combinations)

Expected municipality-age-sex combinations: 12996
Observed municipality-age-sex combinations: 12988
Missing municipality-age-sex combinations: 8


,gemeentenaam,leeftijdsklasse,geslacht
0,RENSWOUDE,90+,M
1,RENSWOUDE,90+,V
2,SCHIERMONNIKOOG,90+,M
3,SCHIERMONNIKOOG,90+,V
4,VLIELAND,5 t/m 9 jaar,V
5,VLIELAND,85 t/m 89 jaar,M
6,VLIELAND,90+,M
7,VLIELAND,90+,V


## 5. Review the Vektis rest category

Vektis uses a rest category for records that cannot be reported in their original municipality–age–sex cells. This includes privacy-protected cells based on fewer than 10 underlying people and records with unknown demographic attributes.

Because these records do not have usable municipality, age, and sex dimensions, they cannot be allocated fairly within a municipality-level age–sex standardisation.

In [8]:
rest_category_mask = (
    df_clean[combination_cols]
    .isna()
    .any(axis=1)
)

rest_category_rows = (
    df_clean
    .loc[rest_category_mask]
    .copy()
)

rest_category_cost = (
    rest_category_rows["total_healthcare_cost"]
    .sum()
)

rest_category_insured_years = (
    rest_category_rows["aantal_verzekerdejaren"]
    .sum()
)

national_total_cost_before_exclusion = (
    df_clean["total_healthcare_cost"]
    .sum()
)

rest_category_cost_share = (
    100
    * rest_category_cost
    / national_total_cost_before_exclusion
)

largest_rest_category_cost_column = (
    rest_category_rows[cost_cols]
    .sum()
    .idxmax()
)

display(
    rest_category_rows[
        [
            "gemeentenaam",
            "leeftijdsklasse",
            "geslacht",
            "aantal_bsn",
            "aantal_verzekerdejaren",
            "total_healthcare_cost",
        ]
    ]
)

rest_category_summary = pd.DataFrame(
    {
        "measure": [
            "Rest-category rows",
            "Healthcare cost (EUR)",
            "Insured years",
            "Share of national healthcare cost (%)",
            "Largest cost column",
        ],
        "value": [
            len(rest_category_rows),
            round(rest_category_cost, 2),
            round(rest_category_insured_years, 2),
            round(rest_category_cost_share, 4),
            largest_rest_category_cost_column,
        ],
    }
)

display(rest_category_summary)

,gemeentenaam,leeftijdsklasse,geslacht,aantal_bsn,aantal_verzekerdejaren,total_healthcare_cost
12908,<NA>,<NA>,<NA>,428642,223608.41,1.331795e+08


,measure,value
0,Rest-category rows,1
1,Healthcare cost (EUR),133179514.59
2,Insured years,223608.41
3,Share of national healthcare cost (%),0.2415
4,Largest cost column,kosten_medisch_specialistische_z


In [9]:
df_final = (
    df_clean
    .loc[~rest_category_mask]
    .reset_index(drop=True)
)

print("Rows before rest-category exclusion:", len(df_clean))
print("Rows available for municipality analysis:", len(df_final))

Rows before rest-category exclusion: 12989
Rows available for municipality analysis: 12988


## 6. Final quality checks

The final analytical dataset is checked for duplicate dimension combinations, invalid insured-year values, and negative healthcare costs before it is saved for the benchmark analysis.

In [10]:
final_quality_checks = {
    "rows_available_for_analysis": len(df_final),
    "columns": df_final.shape[1],
    "municipalities": df_final["gemeentenaam"].nunique(),
    "age_classes": df_final["leeftijdsklasse"].nunique(),
    "sex_categories": df_final["geslacht"].nunique(),
    "expected_dimension_combinations": expected_combination_count,
    "observed_dimension_combinations": observed_combination_count,
    "missing_dimension_combinations": len(missing_combinations),
    "duplicate_dimension_combinations": int(
        df_final.duplicated(combination_cols).sum()
    ),
    "rest_category_rows_excluded": int(rest_category_mask.sum()),
    "rest_category_cost_eur": round(rest_category_cost, 2),
    "rest_category_cost_share_percent": round(
        rest_category_cost_share,
        4,
    ),
    "zero_or_negative_insured_year_rows": int(
        (
            df_final["aantal_verzekerdejaren"]
            <= 0
        ).sum()
    ),
    "negative_total_cost_rows": int(
        (
            df_final["total_healthcare_cost"]
            < 0
        ).sum()
    ),
    "total_healthcare_cost_eur": round(
        df_final["total_healthcare_cost"].sum(),
        2,
    ),
    "total_insured_years": round(
        df_final["aantal_verzekerdejaren"].sum(),
        2,
    ),
}

quality_summary_final = pd.DataFrame.from_dict(
    final_quality_checks,
    orient="index",
    columns=["value"],
)

display(quality_summary_final)

,value
rows_available_for_analysis,1.298800e+04
columns,3.400000e+01
municipalities,3.420000e+02
age_classes,1.900000e+01
sex_categories,2.000000e+00
expected_dimension_combinations,1.299600e+04
observed_dimension_combinations,1.298800e+04
missing_dimension_combinations,8.000000e+00
duplicate_dimension_combinations,0.000000e+00
rest_category_rows_excluded,1.000000e+00


## 7. Save reproducible outputs

The cleaned Parquet file is kept locally and excluded from GitHub. Derived quality-control tables are saved in `outputs/tables` and can be published with the repository.

In [11]:
OUTPUT_TABLES.mkdir(
    parents=True,
    exist_ok=True,
)

DATA_PROCESSED.mkdir(
    parents=True,
    exist_ok=True,
)

quality_summary_path = (
    OUTPUT_TABLES
    / "data_quality_summary.csv"
)

missing_combinations_path = (
    OUTPUT_TABLES
    / "missing_dimension_combinations.csv"
)

rest_category_summary_path = (
    OUTPUT_TABLES
    / "rest_category_summary.csv"
)

processed_data_path = (
    DATA_PROCESSED
    / "vektis_2023_clean.parquet"
)

quality_summary_final.to_csv(
    quality_summary_path,
)

missing_combinations.to_csv(
    missing_combinations_path,
    index=False,
)

rest_category_summary.to_csv(
    rest_category_summary_path,
    index=False,
)

df_final.to_parquet(
    processed_data_path,
    index=False,
)

print("Saved:", quality_summary_path.name)
print("Saved:", missing_combinations_path.name)
print("Saved:", rest_category_summary_path.name)
print("Saved local processed dataset:", processed_data_path.name)

Saved: data_quality_summary.csv
Saved: missing_dimension_combinations.csv
Saved: rest_category_summary.csv
Saved local processed dataset: vektis_2023_clean.parquet


## Data-quality conclusion

After excluding the Vektis rest-category row, the analytical dataset contains **12,988 rows** covering:

- 342 municipalities
- 19 five-year age classes
- 2 sex categories

A fully populated dataset would contain 12,996 municipality–age–sex combinations. Eight combinations are absent from the published data. These combinations are documented but are not imputed, because no defensible values can be inferred from the available aggregated data.

The quality checks found:

- no duplicate municipality–age–sex combinations
- no negative total healthcare costs
- no zero or negative insured-year values
- one Vektis rest-category row without usable municipality, age, or sex dimensions

The rest-category row represents approximately **0.24%** of total healthcare costs. It is excluded from municipality-level standardisation because its costs cannot be assigned fairly to a municipality–age–sex group.

The cleaned local dataset includes three engineered variables:

- `total_healthcare_cost`
- `cost_per_insured_year`
- `specialist_care_cost_per_insured_year`

The dataset is now ready for age–sex-adjusted municipality benchmarking in `02_age_sex_adjusted_benchmarking.ipynb`.